# Customer Life Event Detection — Interactive Colab App (Gradio)
**Author:** Shanmukh Bysani

Run the cell below. A full **interactive UI appears right here in the output** — pick a customer, click Analyse, see detected life events with confidence and product recommendations. You also get a public share link.

Paste your free Groq key when prompted (console.groq.com/keys). Uses synthetic data; production would use a self-hosted LLM.

In [ ]:
# ============================================================================
#  LIFE EVENT DETECTOR  --  interactive Gradio app (runs inline in Colab)
# ============================================================================
!pip install -q groq gradio pandas

import pandas as pd, numpy as np, json, random, re, os, getpass
from datetime import datetime, timedelta
from groq import Groq
import gradio as gr

random.seed(42); np.random.seed(42)

# ---- API key: from environment (Hugging Face Spaces) or prompt (Colab) -------
api_key = os.environ.get("GROQ_API_KEY") or getpass.getpass("Paste your Groq API key: ")
client = Groq(api_key=api_key)
MODEL = "llama-3.3-70b-versatile"   # check console.groq.com/docs/models if it errors

# ---- Synthetic data ----------------------------------------------------------
BASELINE = {
    "Groceries": ["TESCO STORES","SAINSBURYS","ASDA SUPERSTORE","LIDL GB","M&S FOOD"],
    "Dining":    ["JUST EAT","DELIVEROO","GREGGS","PRET A MANGER","COSTA COFFEE"],
    "Transport": ["TFL TRAVEL","UBER","SHELL","BP FUEL","TRAINLINE"],
    "Utilities": ["BRITISH GAS","THAMES WATER","EE MOBILE","SKY DIGITAL"],
    "Shopping":  ["AMAZON UK","ARGOS","JOHN LEWIS"],
}
EVENTS = {
    "home_purchase": ["FOXTONS ESTATE AGENT","CONVEYANCING SOLICITORS LLP","HALIFAX MORTGAGE",
                      "IKEA","DFS FURNITURE","PICKFORDS REMOVALS","B&Q HOME"],
    "new_baby":      ["MOTHERCARE","BOOTS BABY","MAMAS & PAPAS","JOJO MAMAN BEBE",
                      "NHS MATERNITY UNIT","PAMPERS DIRECT"],
    "job_change":    ["CHARLES TYRWHITT","MOO BUSINESS CARDS","LINKEDIN PREMIUM",
                      "REED RECRUITMENT","TM LEWIN FORMAL"],
    "wedding":       ["HATTON GARDEN JEWELLERS","THE WEDDING SHOP","BRIDE & GROOM VENUE",
                      "CONFETTI.CO.UK","WEDDING PHOTOGRAPHY STUDIO"],
}

def generate_customer(profile):
    txns, start = [], datetime(2026,1,1)
    for m in range(3):
        txns.append({"date":(start+timedelta(days=28*m)).strftime("%Y-%m-%d"),
                     "merchant":"SALARY - ACME LTD","amount":round(random.uniform(2800,3500),2)})
    for _ in range(35):
        c=random.choice(list(BASELINE))
        txns.append({"date":(start+timedelta(days=random.randint(0,89))).strftime("%Y-%m-%d"),
                     "merchant":random.choice(BASELINE[c]),"amount":-round(random.uniform(5,120),2)})
    if profile in EVENTS:
        for mch in EVENTS[profile]:
            txns.append({"date":(start+timedelta(days=random.randint(20,85))).strftime("%Y-%m-%d"),
                         "merchant":mch,"amount":-round(random.uniform(150,2500),2)})
    return pd.DataFrame(txns).sort_values("date").reset_index(drop=True)

PROFILES = {
    "Customer A (mystery event)":"home_purchase",
    "Customer B (mystery event)":"new_baby",
    "Customer C (mystery event)":"job_change",
    "Customer D (mystery event)":"normal",
    "Customer E (mystery event)":"wedding",
}

# ---- Privacy + prompt + detection -------------------------------------------
def mask_pii(t):
    t=re.sub(r"\b\d{2}-\d{2}-\d{2}\b","[SORT-CODE]",t); t=re.sub(r"\b\d{8,}\b","[ACCOUNT-NO]",t); return t
def txns_to_text(df):
    return mask_pii("\n".join(f"{r.date} | {r.merchant} | GBP {r.amount:.2f}" for r in df.itertuples()))

SYSTEM=("You are an expert financial behaviour analyst at a UK retail bank. Detect major LIFE EVENTS "
        "from transactions. Be precise; never invent events without evidence.")
def user_prompt(txt):
    return f"""Analyse these transactions and detect life events.
CONSIDER ONLY: home_purchase, new_baby, job_change, wedding, car_purchase
RULES: only report events with supporting transactions; list exact merchant evidence;
give confidence 0.0-1.0; suggest 1-3 banking products per event; empty list if none.

TRANSACTIONS:
{txt}

Respond ONLY with JSON: {{"events":[{{"event":"","confidence":0.0,"evidence":[],"recommended_products":[]}}]}}"""

def detect(df):
    r=client.chat.completions.create(model=MODEL,
        messages=[{"role":"system","content":SYSTEM},
                  {"role":"user","content":user_prompt(txns_to_text(df))}],
        temperature=0.1, response_format={"type":"json_object"})
    raw=r.choices[0].message.content
    try: return json.loads(raw)
    except: return json.loads(raw.replace("```json","").replace("```","").strip())

# ---- Gradio callbacks --------------------------------------------------------
def show_txns(choice):
    return generate_customer(PROFILES[choice])

def analyse(choice):
    try:
        df = generate_customer(PROFILES[choice])
        res = detect(df)
    except Exception as e:
        return (f"### Something went wrong\n```\n{type(e).__name__}: {e}\n```\n"
                f"If this mentions the model, change MODEL to a current name from "
                f"console.groq.com/docs/models (e.g. `llama-3.1-8b-instant`).")

    # The model sometimes returns list items as dicts instead of strings.
    # This helper safely turns any item into clean text.
    def as_text_list(items):
        out_items = []
        for it in items:
            if isinstance(it, dict):
                out_items.append(" - ".join(str(v) for v in it.values()))
            else:
                out_items.append(str(it))
        return out_items

    events = res.get("events", [])
    if not events:
        return "### No clear life event detected for this customer."
    out = ""
    for ev in events:
        conf = float(ev.get("confidence",0))
        bar = "#" * int(conf*20) + "." * (20-int(conf*20))
        out += f"## {ev['event'].replace('_',' ').title()}\n"
        out += f"**Confidence:** `{bar}` {conf:.0%}\n\n"
        out += f"**Evidence:** {', '.join(as_text_list(ev.get('evidence',[])))}\n\n"
        out += "**Recommended products:**\n"
        for p in as_text_list(ev.get("recommended_products",[])):
            out += f"- {p}\n"
        out += "\n---\n"
    return out

# ---- Build the UI ------------------------------------------------------------
with gr.Blocks(title="Life Event Detector") as demo:
    gr.Markdown("# Customer Life Event Detection from Transactions")
    gr.Markdown("Detects life events from transaction history using an LLM. "
                "Synthetic data; production would use a self-hosted LLM for data privacy.")
    with gr.Row():
        with gr.Column():
            pick = gr.Dropdown(list(PROFILES.keys()), value=list(PROFILES.keys())[0],
                               label="Select a customer")
            txn_table = gr.Dataframe(value=generate_customer("home_purchase"),
                                     label="Transaction history")
            btn = gr.Button("Analyse transactions", variant="primary")
        with gr.Column():
            output = gr.Markdown(value="*Click 'Analyse transactions' to detect life events.*")
    pick.change(show_txns, inputs=pick, outputs=txn_table)
    btn.click(analyse, inputs=pick, outputs=output)

# share=True gives a public link; the UI also renders inline below this cell
demo.queue()
demo.launch(share=True, debug=True)